# Lesson 7 - Creating an A2A Sequential Chain Agent with ADK

Now that you have two active agents (Policy Agent and Research Agent), you will orchestrate them. In this lesson, you will use Google ADK's `SequentialAgent` to create a workflow where a user's query is processed by the Research Agent first, and then the Policy Agent. You will use `RemoteA2aAgent` (which acts as A2A Client) to connect to the servers you started in previous lessons.

![Sequential Workflow](sequential.png)

## 7.1. Start the A2A Servers

First, ensure that your `PolicyAgent` server is running.
- Open Terminal 1 by running the cell below.
- If the agent is still running from the previous lesson, you don't need to do anything.
- If the agent has stopped, type: `uv run a2a_policy_agent.py` (you don't need to go back to the previous lesson).

In [2]:
import os

from IPython.display import IFrame, Markdown, display

url_template = os.environ.get("DLAI_LOCAL_URL")
url = url_template.format(port=8888) if url_template else None
# Terminal 1: uv run a2a_policy_agent.py
if url:
    IFrame(f"{url}terminals/1", width=800, height=200)
else:
    display(Markdown("Open a local terminal and run `uv run a2a_policy_agent.py`."))

Open a local terminal and run `uv run a2a_policy_agent.py`.

In [4]:
%cd /content/drive/MyDrive/A2A-MCP

/content/drive/MyDrive/A2A-MCP


In [1]:
%%writefile requirements.txt
google-adk[a2a]==1.19.0
a2a-sdk[http-server]==0.3.16
mcp==1.19.0
agent-framework==1.0.0b251120
agent-framework-a2a==1.0.0b251120
langchain-mcp-adapters
langgraph
langchain[openai]
google-auth==2.42.0
httpx==0.28.1
ipython==8.32.0
nest-asyncio==1.6.0
python-dotenv==1.0.1
requests>=2.32.4,<3.0.0
uvicorn==0.34.0

Overwriting requirements.txt


In [19]:
!uv run a2a_policy_agent.py

Running A2A Health Insurance Policy Agent
INFO:     Started server process [79051]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('::1', 9999, 0, 0): [errno 98] address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
Task exception was never retrieved
future: <Task finished name='Task-1' coro=<Server.serve() done, defined at /content/drive/MyDrive/A2A-MCP/.venv/lib/python3.12/site-packages/uvicorn/server.py:68> exception=SystemExit(1)>
Traceback (most recent call last):
  File "/content/drive/MyDrive/A2A-MCP/.venv/lib/python3.12/site-packages/uvicorn/server.py", line 163, in startup
    server = await loop.create_server(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/asyncio/base_events.py", line 1584, in create_server
    raise OSError(err.errno, msg) from None
OSError: [Errno 98] error while attempting to bind on a

Second, ensure that your Research Agent is running.
- Open Terminal 2 by running the cell below.
- If the agent is still running from the previous lesson, you don't need to do anything.
- If the agent has stopped, type: `uv run a2a_research_agent.py` (you don't need to go back to the previous lesson).

In [8]:
# Terminal 2: uv run a2a_research_agent.py
if url:
    IFrame(f"{url}terminals/2", width=800, height=200)
else:
    display(Markdown("Open another local terminal and run `uv run a2a_research_agent.py`."))

Open another local terminal and run `uv run a2a_research_agent.py`.

In [17]:
!uv run a2a_research_agent.py

Running Health Research Agent
2026-04-14 23:55:37,239 - ERROR - base_events.py:1833 - Task exception was never retrieved
future: <Task finished name='Task-1' coro=<Server.serve() done, defined at /content/drive/MyDrive/A2A-MCP/.venv/lib/python3.12/site-packages/uvicorn/server.py:68> exception=KeyboardInterrupt()>
Traceback (most recent call last):
  File "/content/drive/MyDrive/A2A-MCP/.venv/lib/python3.12/site-packages/uvicorn/main.py", line 579, in run
    server.run()
  File "/content/drive/MyDrive/A2A-MCP/.venv/lib/python3.12/site-packages/uvicorn/server.py", line 66, in run
    return asyncio.run(self.serve(sockets=sockets))
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/drive/MyDrive/A2A-MCP/.venv/lib/python3.12/site-packages/nest_asyncio.py", line 30, in run
    return loop.run_until_complete(task)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/drive/MyDrive/A2A-MCP/.venv/lib/python3.12/site-packages/nest_asyncio.py", line 92, in run_until_comple

## 7.2. Define the Sequential Workflow

Here you will:
1.  Define two `RemoteA2aAgent` instances. These act as A2A clients that know how to communicate with your running servers via A2A.
2.  Create a `SequentialAgent` named `root_agent`. This agent contains the logic to route the workflow through the sub-agents.
3.  Use an `InMemoryRunner` to execute the flow with a specific prompt.


In [10]:
import os

from IPython.display import Markdown, display
from dotenv import load_dotenv
from google.adk.agents import SequentialAgent
from google.adk.agents.remote_a2a_agent import (
    RemoteA2aAgent,
)
from google.adk.runners import InMemoryRunner

import logging
import warnings

logging.disable(level=logging.WARNING)
warnings.filterwarnings("ignore")

In [11]:
load_dotenv()
host = os.environ.get("AGENT_HOST")
policy_port = os.environ.get("POLICY_AGENT_PORT")
research_port = os.environ.get("RESEARCH_AGENT_PORT")

In [12]:
policy_agent = RemoteA2aAgent(
    name="policy_agent",
    agent_card=f"http://{host}:{policy_port}",
)
print("\tℹ️", f"{policy_agent.name} initialized")

	ℹ️ policy_agent initialized


In [13]:
health_research_agent = RemoteA2aAgent(
    name="health_research_agent",
    agent_card=f"http://{host}:{research_port}",
)
print("\tℹ️", f"{health_research_agent.name} initialized")

	ℹ️ health_research_agent initialized


In [14]:
root_agent = SequentialAgent(
    name="root_agent",
    description="Healthcare Routing Agent",
    sub_agents=[
        health_research_agent,
        policy_agent,
    ],
)
print("\tℹ️", f"{root_agent.name} initialized")

	ℹ️ root_agent initialized


## 7.3. Run the Sequential Chain

The `InMemoryRunner` executes the agent. The prompt will trigger the Research Agent to find general info, and then (sequentially) the Policy Agent to check coverage details.

In [15]:
prompt = "How can I get mental health therapy?"

**Note:** It takes a few seconds for the output to display.

In [18]:
print("Running Healthcare Workflow Agent")

runner = InMemoryRunner(root_agent)

for event in await runner.run_debug(prompt, quiet=True):
    if event.is_final_response() and event.content:
        display(Markdown(event.content.parts[0].text))

ERROR:google_adk.google.adk.agents.remote_a2a_agent:Failed to resolve remote A2A agent health_research_agent: Failed to resolve AgentCard from URL http://localhost:9998: HTTP Error 503: Network communication error fetching agent card from http://localhost:9998/.well-known/agent-card.json: All connection attempts failed


Running Healthcare Workflow Agent


You can get mental health therapy through outpatient services. For in-network providers, you will pay 10% coinsurance for office visits and other outpatient services. For out-of-network providers, you will pay 30% coinsurance for office visits and other outpatient services. Inpatient services for mental health, behavioral health, or substance abuse also have a 10% coinsurance for in-network providers and 30% coinsurance for out-of-network providers.

## 7.4. Resources

- [Google ADK Sequential Agents](https://google.github.io/adk-docs/agents/workflow-agents/sequential-agents/)
- [Equivalent notebook in the course repo](https://github.com/holtskinner/A2AWalkthrough/blob/main/5_ADKSequentialAgent.ipynb)